# Pizza Sales Analysis | SQL Project

## Project Overview

This project analyses a year of pizza sales to find where the revenue actually comes from and
where it is being left on the table. It covers the full pipeline:

- data cleaning and validation;
- relational database design (SQLite);
- data population and integrity checks;
- ad-hoc analysis in SQL;
- insights and recommendations.

Everything below runs end to end from the raw CSV. Every table in this notebook is the real
output of the query printed above it, and every number quoted in the Insights section comes
from those outputs.

> **Audited and verified.** This notebook was re-audited end to end in September 2026: the data
> model was rebuilt, the three dataset traps handled in section 4 were fixed, and every figure
> was re-derived from the database and cross-checked against the source CSV. The README carries
> a summary of what changed.

## About the Dataset

48,620 order line items covering the whole of 2015.

| Column | Meaning |
|---|---|
| `pizza_id` | **Line-item** identifier, one per CSV row, *not* a pizza variant |
| `order_id` | Order identifier; one order contains several line items |
| `pizza_name_id` | **Pizza variant** identifier (name + size), e.g. `hawaiian_m` |
| `quantity` | Units of that variant in that order |
| `order_date` | Date the order was placed |
| `order_time` | Time the order was placed |
| `unit_price` | Price of one unit of the variant |
| `total_price` | `quantity * unit_price` for the line item |
| `pizza_size` | S, M, L, XL, XXL |
| `pizza_category` | Classic, Supreme, Veggie, Chicken |
| `pizza_ingredients` | Ingredient list |
| `pizza_name` | Menu name of the pizza |

Source: [Kaggle - Pizza Sales Dataset](https://www.kaggle.com/datasets/nextmillionaire/pizza-sales-dataset)

> The two columns worth being careful with are `pizza_id` and `pizza_name_id`. The names suggest
> the opposite of what they hold: `pizza_id` is unique per row (48,620 values), while the actual
> menu item is `pizza_name_id` (91 values). Section 4.1 verifies this before the schema is built.

## 1. Setup

In [199]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("pizza_sales.db")
CSV_PATH = Path("data/pizza_sales.csv")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 2. Load the Raw Data

In [200]:
raw = pd.read_csv(CSV_PATH)

print(f"{len(raw):,} rows x {raw.shape[1]} columns")
raw.head()

48,620 rows x 12 columns


,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1.0,1.0,hawaiian_m,1.0,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2.0,2.0,classic_dlx_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3.0,2.0,five_cheese_l,1.0,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4.0,2.0,ital_supr_l,1.0,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5.0,2.0,mexicana_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


## 3. First Look at the Data

Nothing is decided in this section. The point is to look at the columns before trusting any of
them, and to let the questions come out of the data rather than out of the column names.

### 3.1 Column profile

In [201]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulls": raw.isna().sum(),
    "distinct": raw.nunique(),
    "distinct_pct": (100 * raw.nunique() / len(raw)).round(1),
})
profile

,dtype,nulls,distinct,distinct_pct
pizza_id,float64,0,48620,100.0
order_id,float64,0,21350,43.9
pizza_name_id,str,0,91,0.2
quantity,float64,0,4,0.0
order_date,str,0,358,0.7
order_time,str,0,16382,33.7
unit_price,float64,0,25,0.1
total_price,float64,0,56,0.1
pizza_size,str,0,5,0.0
pizza_category,str,0,4,0.0


Three things stand out, and none of them are what the column names promised.

- **`pizza_id` is 100% distinct** - one value per row, 48,620 of them. A column that never repeats
  cannot identify a *product*; it identifies a *row*. `pizza_name_id` has 91 values and
  `pizza_name` has 32, so one of those is the real menu key. Which one is settled in section 4.1.
- **`order_date` has 358 distinct values, not 365.** Seven days of the year never appear. That is
  harmless for a total and dangerous for any per-period comparison - section 11 comes back to it.
- **No nulls anywhere**, and `quantity` takes only 4 distinct values. This data is clean in the
  narrow sense. The problems in it are structural, not missing values.

### 3.2 How are the text columns actually formatted?

In [202]:
for col in ["order_date", "order_time"]:
    shapes = raw[col].map(lambda v: re.sub(r"\d", "N", v)).value_counts()
    print(f"{col}:")
    for pattern, n in shapes.items():
        print(f"  {pattern:<12} {n:>6,}")
    print()

order_date:
  NN-NN-NNNN   29,033
  N/N/NNNN     11,284
  NN/N/NNNN     3,656
  N/NN/NNNN     3,604
  NN/NN/NNNN    1,043

order_time:
  NN:NN:NN     48,616
  N:NN:NN           4



`order_date` arrives in **two different shapes**, and pandas will parse both without complaint.
Look at which values take which shape: every slash date has both parts <= 12, every dashed date
starts at 13 or higher. That split is the fingerprint of a spreadsheet that converted the dates
it recognised and left the rest as text - which means the slash dates are ambiguous and the
format cannot be assumed. Section 4.2 settles it on evidence.

`order_time` has 4 rows with no leading zero on the hour. Harmless in pandas, not harmless in
SQLite, where `strftime('%H', '9:52:21')` returns `NULL` rather than an error. The times get
normalised before they are stored.

## 4. Data Cleaning

Three things have to be settled before any table is created: which column identifies a pizza,
how the dates are encoded, and whether `total_price` is worth storing at all.

### 4.1 Which column identifies a pizza?

**Question.** Three columns could plausibly be the menu key: `pizza_id`, `pizza_name_id` and
`pizza_name`. The schema needs exactly one.

**What would settle it.** A dimension key has to repeat across rows - one value per product, not
one per row - and it has to determine every attribute of that product. A candidate that fails
either condition is out.

**Test.** Count distinct values per candidate, then check whether the leading one maps to a single
name, size, category, ingredient list and price.

In [203]:
print("distinct values per candidate key")
for col in ["pizza_id", "order_id", "pizza_name_id", "pizza_name"]:
    print(f"  {col:<14} {raw[col].nunique():>6,}   (rows: {len(raw):,})")

attrs = ["pizza_name", "pizza_size", "pizza_category", "pizza_ingredients", "unit_price"]
inconsistent = raw.groupby("pizza_name_id")[attrs].nunique().gt(1).any(axis=1).sum()
print(f"\npizza_name_id values mapping to more than one set of attributes: {inconsistent}")
print("-> pizza_name_id is the pizza variant key; pizza_id is a line-item id")

distinct values per candidate key
  pizza_id       48,620   (rows: 48,620)
  order_id       21,350   (rows: 48,620)
  pizza_name_id      91   (rows: 48,620)
  pizza_name         32   (rows: 48,620)

pizza_name_id values mapping to more than one set of attributes: 0
-> pizza_name_id is the pizza variant key; pizza_id is a line-item id


**Verdict.** `pizza_name_id` is the menu key: 91 values, each mapping to exactly one set of
attributes. `pizza_id` never repeats, so it identifies a line item rather than a product, and
`pizza_name` (32 values) collapses sizes that are priced separately.

The `pizzas` table is therefore keyed on `pizza_name_id`, and the CSV's `pizza_id` is renamed
`order_detail_id` to say what it actually holds.

### 4.2 Which date format?

**Question.** Section 3.2 found that every slash date has both parts <= 12, so `01/02/2015` could
be 1 February or 2 January. Which is it?

**Two hypotheses.** Either the slash dates are `d/m/Y`, matching the dashed ones, or they are
`m/d/Y`. Both parse without error, so neither can be ruled out by trying it.

**What would settle it.** `order_id` is issued sequentially in time. The correct reading keeps
dates non-decreasing when rows are sorted by `order_id`; the wrong one throws rows backwards.

**Test.** Parse both ways and count how often the date goes backwards.

In [204]:
probe = raw[["order_id", "order_date"]].sort_values("order_id")

for label, dayfirst in [("m/d/Y", False), ("d/m/Y", True)]:
    parsed = pd.to_datetime(probe["order_date"], dayfirst=dayfirst, format="mixed")
    violations = int((parsed.diff().dt.days < 0).sum())
    print(f"{label:>6}: {violations:>3} chronology violations against order_id")

slash_rows = int(raw["order_date"].str.contains("/").sum())
print(f"\n-> the slash dates are d/m/Y; reading them as m/d/Y misdates {slash_rows:,} rows "
      f"({slash_rows / len(raw):.0%})")

 m/d/Y:  22 chronology violations against order_id
 d/m/Y:   0 chronology violations against order_id

-> the slash dates are d/m/Y; reading them as m/d/Y misdates 19,587 rows (40%)


**Verdict.** `d/m/Y`, on zero violations against 22 for the alternative. Both the dashed and the
slash dates are day-first, which is consistent with a spreadsheet that converted the dates it
could read unambiguously and left the rest as text.

Reading them as `m/d/Y` would misdate 19,587 rows and shuffle them between months and weekdays -
silently, since every one of those dates is also valid under the wrong interpretation.

### 4.3 Build the cleaned tables

**Question.** Should `total_price` be stored at all?

**What would settle it.** If it equals `quantity * unit_price` on every row it is derivable, and
storing it only creates a second home for the same number - and a way for the two to disagree
after any later edit.

**Test.** Assert the identity across all 48,620 rows before building anything. If it fails, the
column carries information of its own and has to be kept.

The three tables are then built at one grain each. Nothing else here is a judgement call except
the deduplication of `orders`, which section 5 justifies.

In [205]:
df = raw.copy()
df = df.astype({"order_id": "int64", "quantity": "int64"})

# The slash dates are d/m/Y (section 4.2). Store dates and times as ISO strings so that
# SQLite's date functions and lexicographic ordering both behave.
df["order_date"] = pd.to_datetime(
    df["order_date"], dayfirst=True, format="mixed"
).dt.strftime("%Y-%m-%d")
df["order_time"] = pd.to_datetime(df["order_time"], format="%H:%M:%S").dt.strftime("%H:%M:%S")

# Rename to what the columns actually hold (section 4.1).
df = df.rename(columns={"pizza_name_id": "pizza_id", "pizza_id": "order_detail_id"})

assert (df["quantity"] * df["unit_price"]).round(2).equals(df["total_price"].round(2)), \
    "total_price is not derivable from quantity * unit_price"

# One row per order. The header columns are identical across a whole order, so a JOIN on
# order_id must not be able to multiply rows.
orders = (
    df[["order_id", "order_date", "order_time"]]
    .drop_duplicates(subset="order_id")
    .sort_values("order_id")
    .reset_index(drop=True)
)

# One row per menu item.
pizzas = (
    df[["pizza_id", "pizza_name", "pizza_size", "pizza_category",
        "pizza_ingredients", "unit_price"]]
    .drop_duplicates(subset="pizza_id")
    .sort_values("pizza_id")
    .reset_index(drop=True)
)

# One row per line item.
order_details = (
    df[["order_detail_id", "order_id", "pizza_id", "quantity"]]
    .sort_values("order_detail_id")
    .reset_index(drop=True)
)

for name, table in [("orders", orders), ("pizzas", pizzas), ("order_details", order_details)]:
    print(f"{name:<14} {len(table):>6,} rows")

orders         21,350 rows
pizzas             91 rows
order_details  48,620 rows


**Verdict.** The assertion passes, so `total_price` is dropped and revenue is derived in the
queries instead. `unit_price` moves to `pizzas`, where it belongs: every variant carries exactly
one price across the whole year.

## 5. Why the Grain Matters

`orders` above was deduplicated down to one row per order. That step is easy to skip: the CSV
already carries `order_id`, `order_date` and `order_time` on every row, so a table built straight
from those three columns looks finished.

It is not, and nothing about it complains. Here is what taking the CSV at face value does.

In [206]:
# One "orders" row per CSV row - no deduplication.
naive_orders = raw[["order_id", "order_date", "order_time"]]
lines = raw[["order_id", "pizza_category", "quantity", "total_price"]]

# Revenue by category needs no join: the category is already on the line.
by_category = lines.groupby("pizza_category")["total_price"].sum().sum()

# Revenue by hour needs the order time, so it has to join to orders.
fanned = lines.merge(naive_orders, on="order_id")
hour = pd.to_datetime(fanned["order_time"], format="%H:%M:%S").dt.hour
by_hour = fanned.groupby(hour)["total_price"].sum().sum()

print(f"rows before the join : {len(lines):>9,}")
print(f"rows after the join  : {len(fanned):>9,}")
print()
print(f"revenue summed by category : ${by_category:>12,.2f}")
print(f"revenue summed by hour     : ${by_hour:>12,.2f}")
print(f"ratio                      : {by_hour / by_category:>12.2f}x")

rows before the join :    48,620
rows after the join  :   172,836

revenue summed by category : $  817,860.05
revenue summed by hour     : $2,951,196.95
ratio                      :         3.61x


The same dataset yields two different annual revenue figures, 3.6x apart. Both aggregations are
ordinary; neither raises anything.

The row count says why. `order_id` is not unique in `naive_orders`, so each line item matches
once per line in its own order:

In [207]:
sample = raw[raw["order_id"] == 2]
n_lines = len(sample)
value = sample["total_price"].sum()

print(f"order 2 holds {n_lines} line items worth ${value:,.2f}")
print(f"naive_orders holds {(naive_orders['order_id'] == 2).sum()} rows for order 2")
print(f"the join returns {n_lines * n_lines} rows worth ${value * n_lines:,.2f}")
print()
print("-> every order is multiplied by its own line count, so orders with more items are")
print("   inflated more. That is why the distortion is uneven across hours and weekdays,")
print("   rather than a constant factor that might have been spotted.")

order 2 holds 5 line items worth $92.00
naive_orders holds 5 rows for order 2
the join returns 25 rows worth $460.00

-> every order is multiplied by its own line count, so orders with more items are
   inflated more. That is why the distortion is uneven across hours and weekdays,
   rather than a constant factor that might have been spotted.


Two things worth carrying forward:

1. **The category and size breakdowns come out correct**, because they never join to `orders`.
   A report built this way therefore disagrees with itself, and that disagreement is the only
   symptom on offer.
2. **The row count is the cheap test.** Joining a fact table to its dimensions must never increase
   the row count. Section 9 asserts exactly that, on every load.

Making `order_id` the primary key of `orders` turns a silent error into an impossible one: the
duplicate rows cannot be inserted at all.

## 6. Database Design

Three tables, one grain each:

- **`orders`** - one row per order (`order_id`, date, time). 21,350 rows.
- **`pizzas`** - one row per menu item, keyed on `pizza_id` (`hawaiian_m`), carrying name, size,
  category, ingredients and price. 91 rows.
- **`order_details`** - the fact table, one row per line item, linking an order to a pizza with a
  quantity. 48,620 rows.

Revenue is always computed as `order_details.quantity * pizzas.unit_price`.

`order_details` is the fact table and sits on the "many" side of both relationships: an order
*contains* its lines and owns them, while a pizza is merely *referenced* by a line and exists on
the menu whether or not anyone orders it.

The constraints that matter:

- `orders.order_id` is the primary key, so `orders` cannot contain a duplicate order. Joining
  `order_details` to an `orders` table that still has one row per *line item* silently multiplies
  every revenue figure by the number of items in the order.
- `order_details` is `UNIQUE (order_id, pizza_id)`, so the same variant cannot be recorded twice
  in one order.
- `pizzas.pizza_id` is declared `TEXT NOT NULL PRIMARY KEY`. SQLite keeps a legacy quirk where a
  non-INTEGER `PRIMARY KEY` column still accepts `NULL`, so the `NOT NULL` is not redundant here.
- All three tables are `STRICT`, so SQLite rejects a value of the wrong type instead of quietly
  coercing it - the text `'three'` cannot end up in `quantity`.

## 7. Create the Schema

In [208]:
SCHEMA = """
DROP TABLE IF EXISTS order_details;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS pizzas;

CREATE TABLE orders (
    order_id   INTEGER PRIMARY KEY,      -- one order placed by a customer
    order_date TEXT NOT NULL,            -- date of the order, ISO 'YYYY-MM-DD'
    order_time TEXT NOT NULL             -- time of the order, 'HH:MM:SS'
) STRICT;

CREATE TABLE pizzas (
    -- A plain TEXT PRIMARY KEY still accepts NULL in SQLite, so NOT NULL is explicit.
    pizza_id          TEXT NOT NULL PRIMARY KEY,  -- menu item: name + size, e.g. 'hawaiian_m'
    pizza_name        TEXT NOT NULL,              -- menu name, e.g. 'The Hawaiian Pizza'
    pizza_size        TEXT NOT NULL CHECK (pizza_size IN ('S', 'M', 'L', 'XL', 'XXL')),
    pizza_category    TEXT NOT NULL,              -- Classic, Supreme, Veggie or Chicken
    pizza_ingredients TEXT NOT NULL,              -- comma-separated ingredient list
    unit_price        REAL NOT NULL CHECK (unit_price > 0)   -- price of one unit, USD
) STRICT;

CREATE TABLE order_details (
    order_detail_id INTEGER PRIMARY KEY,  -- one line on one order
    order_id        INTEGER NOT NULL REFERENCES orders(order_id),  -- the order it belongs to
    pizza_id        TEXT    NOT NULL REFERENCES pizzas(pizza_id),  -- the menu item ordered
    quantity        INTEGER NOT NULL CHECK (quantity > 0),         -- units on this line
    UNIQUE (order_id, pizza_id)
) STRICT;

CREATE INDEX idx_order_details_order_id ON order_details(order_id);
CREATE INDEX idx_order_details_pizza_id ON order_details(pizza_id);
"""

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON")   # SQLite does not enforce FKs unless asked
con.executescript(SCHEMA)

print(pd.read_sql_query(
    "SELECT type, name FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type, name",
    con,
).to_string(index=False))

 type                             name
index       idx_order_details_order_id
index       idx_order_details_pizza_id
index sqlite_autoindex_order_details_1
index        sqlite_autoindex_pizzas_1
table                    order_details
table                           orders
table                           pizzas


## 8. Load the Data

In [209]:
# Clear the tables first so this cell can be re-run on its own without colliding with the
# rows it inserted last time. Children before parents, because foreign keys are enforced.
for table in ("order_details", "orders", "pizzas"):
    con.execute(f"DELETE FROM {table}")

orders.to_sql("orders", con, if_exists="append", index=False)
pizzas.to_sql("pizzas", con, if_exists="append", index=False)
order_details.to_sql("order_details", con, if_exists="append", index=False)
con.commit()

print(pd.read_sql_query(
    """
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
    UNION ALL SELECT 'pizzas',        COUNT(*) FROM pizzas
    UNION ALL SELECT 'order_details', COUNT(*) FROM order_details
    """,
    con,
).to_string(index=False))

   table_name  row_count
       orders      21350
       pizzas         91
order_details      48620


## 9. Data Quality Checks

The fan-out check is the important one. A join that returns more rows than the fact table has is
the classic way to inflate every aggregate downstream, and it fails silently: the query still
runs and the numbers still look plausible.

In [210]:
checks = {}

checks["orders.order_id is unique"] = con.execute(
    "SELECT COUNT(*) = COUNT(DISTINCT order_id) FROM orders"
).fetchone()[0]

checks["pizzas holds 91 variants"] = con.execute(
    "SELECT COUNT(*) = 91 FROM pizzas"
).fetchone()[0]

checks["no orphan foreign keys"] = len(con.execute("PRAGMA foreign_key_check").fetchall()) == 0

checks["no order without line items"] = con.execute(
    """
    SELECT COUNT(*) = 0 FROM orders AS o
    WHERE NOT EXISTS (SELECT 1 FROM order_details AS od WHERE od.order_id = o.order_id)
    """
).fetchone()[0]

checks["join does not fan out"] = con.execute(
    """
    SELECT COUNT(*) = (SELECT COUNT(*) FROM order_details)
    FROM order_details AS od
    JOIN orders AS o ON o.order_id = od.order_id
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    """
).fetchone()[0]

db_revenue = con.execute(
    """
    SELECT SUM(od.quantity * p.unit_price)
    FROM order_details AS od
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    """
).fetchone()[0]
checks["revenue matches the CSV"] = round(db_revenue, 2) == round(raw["total_price"].sum(), 2)

for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

assert all(checks.values()), "data quality checks failed"

[PASS] orders.order_id is unique
[PASS] pizzas holds 91 variants
[PASS] no orphan foreign keys
[PASS] no order without line items
[PASS] join does not fan out
[PASS] revenue matches the CSV


## 10. Ad-hoc Analysis

Every query below is executed against the database; the table underneath each cell is its actual
result set.

In [211]:
def run(sql: str) -> pd.DataFrame:
    """Execute a query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con)

### 10.1 Headline Numbers

In [212]:
run("""
SELECT
    COUNT(DISTINCT o.order_id)                AS orders,
    SUM(od.quantity)                          AS pizzas_sold,
    ROUND(SUM(od.quantity * p.unit_price), 2) AS total_revenue,
    COUNT(DISTINCT o.order_date)              AS days_with_sales,
    MIN(o.order_date)                         AS first_day,
    MAX(o.order_date)                         AS last_day
FROM
    orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id;
""")

,orders,pizzas_sold,total_revenue,days_with_sales,first_day,last_day
0,21350,49574,817860.05,358,2015-01-01,2015-12-31


**Reads as:** a full calendar year, 2015-01-01 to 2015-12-31 - but only 358 days carry sales. That
is the seven-day gap section 3.1 flagged, confirmed here rather than explained.

**Does not say:** anything about distribution. 49,574 pizzas across 21,350 orders averages 2.32 per
order, and most of the sections below exist because that average hides the shape.

### 10.2 Average Sales per Day

In [213]:
run("""
SELECT
    ROUND(AVG(daily_revenue), 2) AS avg_daily_revenue,
    ROUND(AVG(daily_pizzas), 2)  AS avg_daily_pizzas,
    COUNT(*)                     AS days_with_sales
FROM (
    SELECT
        o.order_date,
        SUM(od.quantity * p.unit_price) AS daily_revenue,
        SUM(od.quantity)                AS daily_pizzas
    FROM
        orders AS o
        JOIN order_details AS od ON od.order_id = o.order_id
        JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
    GROUP BY o.order_date
);
""")

,avg_daily_revenue,avg_daily_pizzas,days_with_sales
0,2284.53,138.47,358


**Reads as:** \$2,284.53 per trading day, averaged over the 358 days that actually traded.

**Why not divide by 365:** that would give \$2,240.71 and quietly credit the shop with seven days
it never worked. Averaging over days that exist is the only honest denominator here.

**Does not say:** whether days resemble each other. Section 10.7 shows Friday running 43% ahead of
Sunday per trading day.

### 10.3 Average Order Size

In [214]:
run("""
SELECT
    ROUND(AVG(order_pizzas), 2) AS avg_pizzas_per_order,
    ROUND(AVG(order_value), 2)  AS avg_order_value
FROM (
    SELECT
        od.order_id,
        SUM(od.quantity)                AS order_pizzas,
        SUM(od.quantity * p.unit_price) AS order_value
    FROM
        order_details   AS od
        JOIN pizzas     AS p ON p.pizza_id = od.pizza_id
    GROUP BY od.order_id
);
""")

,avg_pizzas_per_order,avg_order_value
0,2.32,38.31


**Reads as:** a typical order is 2.32 pizzas worth \$38.31.

**Does not say** how typical. The median order is 2 pizzas and the largest is 28, so the mean sits
above the middle of a skewed distribution. Useful for capacity planning, misleading as a portrait
of a normal customer.

### 10.4 Top 5 Best- and Worst-Selling Pizzas

In [215]:
run("""
WITH pizza_sales AS (
    SELECT
        p.pizza_name,
        SUM(od.quantity) AS units_sold
    FROM
        pizzas AS p
        JOIN order_details AS od ON od.pizza_id = p.pizza_id
    GROUP BY p.pizza_name
),
ranked AS (
    SELECT
        pizza_name,
        units_sold,
        ROW_NUMBER() OVER (ORDER BY units_sold DESC) AS rank_best,
        ROW_NUMBER() OVER (ORDER BY units_sold ASC)  AS rank_worst
    FROM
        pizza_sales
)
SELECT
    best.rank_best   AS rank,
    best.pizza_name  AS best_selling_pizza,
    best.units_sold  AS best_units_sold,
    worst.pizza_name AS worst_selling_pizza,
    worst.units_sold AS worst_units_sold
FROM
    ranked AS best
    JOIN ranked AS worst ON worst.rank_worst = best.rank_best
WHERE best.rank_best <= 5
ORDER BY best.rank_best;
""")

,rank,best_selling_pizza,best_units_sold,worst_selling_pizza,worst_units_sold
0,1,The Classic Deluxe Pizza,2453,The Brie Carre Pizza,490
1,2,The Barbecue Chicken Pizza,2432,The Mediterranean Pizza,934
2,3,The Hawaiian Pizza,2422,The Calabrese Pizza,937
3,4,The Pepperoni Pizza,2418,The Spinach Supreme Pizza,950
4,5,The Thai Chicken Pizza,2371,The Soppressata Pizza,961


**Reads as:** a 5:1 gap between the best and the worst seller, with The Brie Carre roughly half
the next-worst.

**Does not say** why. The ranking is by units summed across sizes, and the pizzas are not all
offered in the same size range - section 10.13 checks whether availability or price explains the
bottom of this table.

### 10.5 Most Profitable Hours of the Day

In [216]:
run("""
SELECT
    strftime('%H:00', o.order_time)                        AS hour_of_day,
    SUM(od.quantity)                                       AS total_quantity,
    ROUND(AVG(SUM(od.quantity)) OVER ())                   AS avg_quantity,
    ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
    ROUND(AVG(SUM(od.quantity * p.unit_price)) OVER (), 2) AS avg_revenue
FROM
    orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY hour_of_day
ORDER BY total_revenue DESC;
""")

,hour_of_day,total_quantity,avg_quantity,total_revenue,avg_revenue
0,12:00,6776,3305.0,111877.90,54524.0
1,13:00,6413,3305.0,106065.70,54524.0
2,18:00,5417,3305.0,89296.85,54524.0
3,17:00,5211,3305.0,86237.45,54524.0
4,19:00,4406,3305.0,72628.90,54524.0
5,16:00,4239,3305.0,70055.40,54524.0
6,14:00,3613,3305.0,59201.40,54524.0
7,20:00,3534,3305.0,58215.40,54524.0
8,15:00,3216,3305.0,52992.30,54524.0
9,11:00,2728,3305.0,44935.80,54524.0


**Reads as:** two peaks - lunch and dinner - with 12:00 alone worth \$111,878.

**Read the average line carefully.** The \$54,524 figure includes 09:00, 10:00 and 23:00, which
barely trade at all. Across the twelve hours that genuinely operate the average is \$68,029, so
the line on the chart sits about 20% below a real trading hour rather than marking one.

### 10.6 The Quiet Hours (09:00, 10:00, 23:00), Month by Month

In [217]:
run("""
SELECT
    strftime('%Y-%m', o.order_date)           AS month,
    strftime('%H:00', o.order_time)           AS hour_of_day,
    COUNT(DISTINCT o.order_id)                AS orders,
    SUM(od.quantity)                          AS total_quantity,
    ROUND(SUM(od.quantity * p.unit_price), 2) AS total_revenue
FROM
    orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
WHERE strftime('%H', o.order_time) IN ('09', '10', '23')
GROUP BY month, hour_of_day
ORDER BY month, hour_of_day;
""")

,month,hour_of_day,orders,total_quantity,total_revenue
0,2015-01,23:00,1,2,31.25
1,2015-02,10:00,1,3,47.90
2,2015-02,23:00,2,6,101.50
3,2015-03,10:00,1,3,50.25
4,2015-03,23:00,1,3,40.75
5,2015-04,10:00,1,3,52.75
6,2015-04,23:00,2,4,64.50
7,2015-05,10:00,1,1,20.75
8,2015-05,23:00,1,1,16.50
9,2015-06,10:00,1,2,28.75


**Reads as:** 23:00 trades in all twelve months, 10:00 in eight, 09:00 in exactly one.

**So the three are not the same thing.** Only 23:00 is a quiet hour. The other two are hours the
shop effectively does not trade in, and 09:00 is a single order for the entire year - no rate
computed from it means anything.

### 10.7 Profitability by Day of the Week

In [218]:
run("""
SELECT
    CASE strftime('%w', o.order_date)
        WHEN '0' THEN 'Sunday'   WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'  WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
        ELSE 'Saturday'
    END                                                    AS day_of_week,
    COUNT(DISTINCT o.order_date)                           AS operating_days,
    COUNT(DISTINCT o.order_id)                             AS total_orders,
    SUM(od.quantity)                                       AS total_quantity,
    ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
    ROUND(SUM(od.quantity * p.unit_price)
        / COUNT(DISTINCT o.order_date), 2)                 AS revenue_per_day,
    ROUND(AVG(SUM(od.quantity * p.unit_price)) OVER (), 2) AS avg_revenue
FROM
    orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY day_of_week
ORDER BY total_revenue DESC;
""")

,day_of_week,operating_days,total_orders,total_quantity,total_revenue,revenue_per_day,avg_revenue
0,Friday,50,3538,8242,136073.90,2721.48,116837.15
1,Thursday,52,3239,7478,123528.50,2375.55,116837.15
2,Saturday,52,3158,7493,123182.40,2368.89,116837.15
3,Wednesday,52,3024,6946,114408.40,2200.16,116837.15
4,Tuesday,52,2973,6895,114133.80,2194.88,116837.15
5,Monday,48,2794,6485,107329.55,2236.03,116837.15
6,Sunday,52,2624,6035,99203.50,1907.76,116837.15


**Reads as:** Friday best and Sunday worst, on totals *and* on revenue per trading day. Rankings
that agree on both measures are safe to act on.

**Watch the `operating_days` column.** Monday traded 48 days against 52 for most of the week,
which is why its two rankings disagree - 6th on totals, 4th per trading day. Section 11 follows
that thread to its source.

### 10.8 Days With No Sales

`revenue_per_day` in the previous query is not cosmetic: the calendar is not complete, and the
gaps are not spread evenly across weekdays.

In [219]:
run("""
WITH RECURSIVE calendar(day) AS (
    SELECT '2015-01-01'
    UNION ALL
    SELECT date(day, '+1 day') FROM calendar WHERE day < '2015-12-31'
)
SELECT
    c.day AS missing_day,
    CASE strftime('%w', c.day)
        WHEN '0' THEN 'Sunday'   WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'  WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
        ELSE 'Saturday'
    END   AS day_of_week
FROM
    calendar AS c
WHERE NOT EXISTS (SELECT 1 FROM orders AS o WHERE o.order_date = c.day)
ORDER BY c.day;
""")

,missing_day,day_of_week
0,2015-09-24,Thursday
1,2015-09-25,Friday
2,2015-10-05,Monday
3,2015-10-12,Monday
4,2015-10-19,Monday
5,2015-10-26,Monday
6,2015-12-25,Friday


**Reads as:** seven missing days, four of them October Mondays.

That is not a distribution you get by chance, and it is why section 11 exists.

**Does not say** *why* they are missing. A deliberate closure and a recording failure look
identical from here, and the difference decides whether anything should be done about it.

### 10.9 Profitability by Month

In [220]:
run("""
SELECT
    strftime('%Y-%m', o.order_date)                             AS month,
    COUNT(DISTINCT o.order_date)                                AS operating_days,
    COUNT(DISTINCT o.order_id)                                  AS total_orders,
    SUM(od.quantity)                                            AS total_quantity,
    ROUND(SUM(od.quantity * p.unit_price), 2)                   AS total_revenue,
    ROUND(AVG(SUM(od.quantity * p.unit_price)) OVER (), 2)      AS avg_revenue,
    RANK() OVER (ORDER BY SUM(od.quantity * p.unit_price) DESC) AS rank_by_revenue
FROM
    orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY month
ORDER BY month;
""")

,month,operating_days,total_orders,total_quantity,total_revenue,avg_revenue,rank_by_revenue
0,2015-01,31,1845,4232,69793.30,68155.0,5
1,2015-02,28,1685,3961,65159.60,68155.0,9
2,2015-03,31,1840,4261,70397.10,68155.0,3
3,2015-04,30,1799,4151,68736.80,68155.0,6
4,2015-05,31,1853,4328,71402.75,68155.0,2
5,2015-06,30,1773,4107,68230.20,68155.0,8
6,2015-07,31,1935,4392,72557.90,68155.0,1
7,2015-08,31,1841,4168,68278.25,68155.0,7
8,2015-09,28,1661,3890,64180.05,68155.0,11
9,2015-10,27,1646,3883,64027.60,68155.0,12


**Reads as:** a flat year - \$64,028 to \$72,558 - with October last.

**Does not say** that October sold least. The `operating_days` column differs across months and
these totals do not account for it. Section 11 settles the question.

### 10.10 Performance by Pizza Category

In [221]:
run("""
SELECT
    p.pizza_category,
    SUM(od.quantity)                                       AS total_quantity,
    ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
    ROUND(100.0 * SUM(od.quantity * p.unit_price)
        / SUM(SUM(od.quantity * p.unit_price)) OVER (), 1) AS revenue_share_pct
FROM
    pizzas AS p
    JOIN order_details AS od ON od.pizza_id = p.pizza_id
GROUP BY p.pizza_category
ORDER BY total_quantity DESC;
""")

,pizza_category,total_quantity,total_revenue,revenue_share_pct
0,Classic,14888,220053.10,26.9
1,Supreme,11987,208197.00,25.5
2,Veggie,11649,193690.45,23.7
3,Chicken,11050,195919.50,24.0


**Reads as:** four categories within 3.2 percentage points of each other on revenue share -
effectively a tie.

Classic leads on units by a wider margin than on revenue because it is the cheapest category per
pizza sold (\$14.78 against \$17.73 for Chicken). Category is not where this menu differentiates.

### 10.11 Performance by Pizza Size

In [222]:
run("""
SELECT
    p.pizza_size,
    COUNT(DISTINCT p.pizza_name)                           AS pizzas_offered,
    SUM(od.quantity)                                       AS total_quantity,
    ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
    ROUND(100.0 * SUM(od.quantity * p.unit_price)
        / SUM(SUM(od.quantity * p.unit_price)) OVER (), 1) AS revenue_share_pct
FROM
    pizzas AS p
    JOIN order_details AS od ON od.pizza_id = p.pizza_id
GROUP BY p.pizza_size
ORDER BY total_quantity DESC;
""")

,pizza_size,pizzas_offered,total_quantity,total_revenue,revenue_share_pct
0,L,30,18956,375318.70,45.9
1,M,29,15635,249382.25,30.5
2,S,30,14403,178076.50,21.8
3,XL,1,552,14076.00,1.7
4,XXL,1,28,1006.60,0.1


**Reads as:** L alone is 45.9% of revenue; XL and XXL together are 1.8%.

**Does not say** that customers reject the large formats. The `pizzas_offered` column shows XL and
XXL each exist on a single pizza, against 30 for S and L. Low share and low availability are
confounded here, and this dataset cannot separate them.

### 10.12 Top 5 Orders by Value

In [223]:
run("""
SELECT
    o.order_id,
    o.order_date,
    o.order_time,
    SUM(od.quantity)                          AS pizzas,
    ROUND(SUM(od.quantity * p.unit_price), 2) AS order_total
FROM
    orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY o.order_id
ORDER BY order_total DESC
LIMIT 5;
""")

,order_id,order_date,order_time,pizzas,order_total
0,18845,2015-11-18,12:25:12,28,444.20
1,10760,2015-06-30,13:31:27,25,417.15
2,1096,2015-01-19,12:56:45,15,285.15
3,6169,2015-04-14,13:14:51,15,284.00
4,740,2015-01-13,12:29:51,15,280.95


**Reads as:** the largest order is \$444.20 and 28 pizzas - 11.6x the average order value.

**Worth noticing:** all five were placed between 12:25 and 13:31. The biggest orders land inside
the lunch peak, in the same hour the kitchen is already most loaded.

### 10.13 Menu Structure: Sizes Offered and Entry Price

Sorted slowest-selling first. `entry_price` is the cheapest size a customer can buy the pizza in
- the price of entry to that item.

In [224]:
run("""
WITH menu AS (
    SELECT
        p.pizza_name,
        COUNT(DISTINCT p.pizza_size)        AS sizes_offered,
        GROUP_CONCAT(DISTINCT p.pizza_size) AS sizes,
        MIN(p.unit_price)                   AS entry_price,
        SUM(od.quantity)                    AS units_sold
    FROM
        pizzas AS p
        JOIN order_details AS od ON od.pizza_id = p.pizza_id
    GROUP BY p.pizza_name
)
SELECT
    pizza_name,
    sizes_offered,
    sizes,
    entry_price,
    ROUND(AVG(entry_price) OVER (), 2) AS menu_avg_entry_price,
    units_sold,
    RANK() OVER (ORDER BY units_sold)  AS slowest_rank
FROM
    menu
ORDER BY units_sold;
""")

,pizza_name,sizes_offered,sizes,entry_price,menu_avg_entry_price,units_sold,slowest_rank
0,The Brie Carre Pizza,1,S,23.65,12.79,490,1
1,The Mediterranean Pizza,3,"M,L,S",12.00,12.79,934,2
2,The Calabrese Pizza,3,"M,S,L",12.25,12.79,937,3
3,The Spinach Supreme Pizza,3,"S,M,L",12.50,12.79,950,4
4,The Soppressata Pizza,3,"L,M,S",12.50,12.79,961,5
5,The Spinach Pesto Pizza,3,"L,S,M",12.50,12.79,970,6
6,The Chicken Pesto Pizza,3,"L,M,S",12.75,12.79,973,7
7,The Italian Vegetables Pizza,3,"S,L,M",12.75,12.79,981,8
8,The Chicken Alfredo Pizza,3,"S,M,L",12.75,12.79,987,9
9,The Green Garden Pizza,3,"S,L,M",12.00,12.79,997,10


**Reads as:** 27 of the 32 pizzas are sold in S, M and L at an entry price near \$12. The menu is
far more uniform than the sales figures suggest.

**This settles the question left open at 10.4.** The Brie Carre's single-size listing does not
explain its volume, because The Big Meat is also single-size and sells nearly four times as much.
Its entry price of \$23.65 against a menu average of \$12.79 is the better candidate.

### 10.14 Line Items of the Largest Order

In [225]:
run("""
WITH biggest_order AS (
    SELECT od.order_id
    FROM
        order_details AS od
        JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    GROUP BY od.order_id
    ORDER BY SUM(od.quantity * p.unit_price) DESC
    LIMIT 1
)
SELECT
    p.pizza_name,
    p.pizza_size,
    od.quantity,
    p.unit_price,
    ROUND(od.quantity * p.unit_price, 2)                          AS line_total,
    ROUND(SUM(od.quantity * p.unit_price) OVER (
        ORDER BY od.quantity * p.unit_price DESC, p.pizza_id), 2) AS running_total,
    RANK() OVER (ORDER BY od.quantity * p.unit_price DESC)        AS line_rank
FROM
    order_details AS od
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
WHERE od.order_id = (SELECT order_id FROM biggest_order)
ORDER BY line_rank;
""")

,pizza_name,pizza_size,quantity,unit_price,line_total,running_total,line_rank
0,The Barbecue Chicken Pizza,M,3,16.75,50.25,50.25,1
1,The Thai Chicken Pizza,L,2,20.75,41.50,91.75,2
2,The Sicilian Pizza,M,2,16.25,32.50,124.25,3
3,The Greek Pizza,XL,1,25.50,25.50,149.75,4
4,The Pepperoni Pizza,M,2,12.50,25.00,174.75,5
5,The Big Meat Pizza,S,2,12.00,24.00,198.75,6
6,The Hawaiian Pizza,S,2,10.50,21.00,219.75,7
7,The Prosciutto and Arugula Pizza,L,1,20.75,20.75,240.50,8
8,The Southwest Chicken Pizza,L,1,20.75,20.75,261.25,8
9,The Vegetables + Vegetables Pizza,L,1,20.25,20.25,281.50,10


**Reads as:** 21 separate line items, 28 pizzas, almost all in quantities of one or two.

The running total climbs evenly with no single item dominating, so this is a party or office order
assembled across the menu rather than a bulk purchase of one product. A curiosity, not a segment
worth planning around.

## 11. Investigation: Is October Genuinely the Weakest Month?

**Question.** Section 10.9 puts October last on revenue for the year. That is a fact about a total, not yet a
fact about demand - and section 3.1 already noted that seven days are missing from the data.

**Two hypotheses** fit the same number equally well:

- **Demand.** October really did sell less on the days it traded.
- **Coverage.** October is summed over fewer days than the months it is being compared against.

They point at opposite actions - a promotion in one case, a data fix in the other - so they have
to be told apart before anything is recommended.

In [226]:
# Reused by every query in this section, so it is defined once.
MONTHLY = """
WITH monthly AS (
    SELECT
        strftime('%Y-%m', o.order_date) AS month,
        COUNT(DISTINCT o.order_date)    AS trading_days,
        SUM(od.quantity * p.unit_price) AS revenue
    FROM
        orders AS o
        JOIN order_details AS od ON od.order_id = o.order_id
        JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
    GROUP BY month
)
"""

### 11.1 How many days did each month actually trade?

Comparing recorded trading days against the calendar. If coverage is even, every month shows
zero missing days and the demand explanation survives.

In [227]:
run(MONTHLY + """
SELECT
    month,
    CAST(strftime('%d', date(month || '-01', '+1 month', '-1 day')) AS INT) AS calendar_days,
    trading_days,
    CAST(strftime('%d', date(month || '-01', '+1 month', '-1 day')) AS INT)
        - trading_days                                                      AS missing_days,
    ROUND(revenue, 2)                                                       AS revenue
FROM
    monthly
ORDER BY missing_days DESC, month;
""")

,month,calendar_days,trading_days,missing_days,revenue
0,2015-10,31,27,4,64027.60
1,2015-09,30,28,2,64180.05
2,2015-12,31,30,1,64701.15
3,2015-01,31,31,0,69793.30
4,2015-02,28,28,0,65159.60
5,2015-03,31,31,0,70397.10
6,2015-04,30,30,0,68736.80
7,2015-05,31,31,0,71402.75
8,2015-06,30,30,0,68230.20
9,2015-07,31,31,0,72557.90


Coverage is not even. The three lowest-revenue months of the year are exactly the three months
with missing days, and they are missing them in that order: October 4, September 2, December 1.
Every other month is complete.

That is close enough to a one-to-one match to be suspicious. The next question is whether the
missing days are random.

### 11.2 Which days are missing, and do they fall in a pattern?

In [228]:
run("""
WITH RECURSIVE calendar(day) AS (
    SELECT '2015-01-01'
    UNION ALL
    SELECT date(day, '+1 day') FROM calendar WHERE day < '2015-12-31'
)
SELECT
    c.day AS missing_day,
    CASE strftime('%w', c.day)
        WHEN '0' THEN 'Sunday'   WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'  WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
        ELSE 'Saturday'
    END   AS day_of_week
FROM
    calendar AS c
WHERE NOT EXISTS (SELECT 1 FROM orders AS o WHERE o.order_date = c.day)
ORDER BY c.day;
""")

,missing_day,day_of_week
0,2015-09-24,Thursday
1,2015-09-25,Friday
2,2015-10-05,Monday
3,2015-10-12,Monday
4,2015-10-19,Monday
5,2015-10-26,Monday
6,2015-12-25,Friday


Not random. **All four missing October days are Mondays** - every Monday the month had. September
loses a Thursday and a Friday, December loses 25 December alone, which reads like a deliberate
closure rather than a gap.

So October is not being measured on a smaller random sample. It is missing four specific shifts,
and the correction is to compare per trading day instead of per month.

### 11.3 Ranking on revenue per trading day

In [229]:
run(MONTHLY + """
SELECT
    month,
    trading_days,
    ROUND(revenue, 2)                                  AS revenue,
    RANK() OVER (ORDER BY revenue DESC)                AS rank_by_total,
    ROUND(revenue / trading_days, 2)                   AS revenue_per_trading_day,
    RANK() OVER (ORDER BY revenue / trading_days DESC) AS rank_per_trading_day
FROM
    monthly
ORDER BY rank_per_trading_day;
""")

,month,trading_days,revenue,rank_by_total,revenue_per_trading_day,rank_per_trading_day
0,2015-10,27,64027.60,12,2371.39,1
1,2015-11,30,70395.35,4,2346.51,2
2,2015-07,31,72557.90,1,2340.58,3
3,2015-02,28,65159.60,9,2327.13,4
4,2015-05,31,71402.75,2,2303.31,5
5,2015-09,28,64180.05,11,2292.14,6
6,2015-04,30,68736.80,6,2291.23,7
7,2015-06,30,68230.20,8,2274.34,8
8,2015-03,31,70397.10,3,2270.87,9
9,2015-01,31,69793.30,5,2251.40,10


### Verdict

The conclusion inverts. **October moves from 12th on totals to 1st on revenue per trading day** -
it was not the weakest month of 2015, it was the strongest, and it only looked weak because four
Mondays are absent from the data. September moves from 11th to 6th, into the middle of the pack.

**December is the month that stays down**: 10th on totals and 12th per trading day. It is the
only one of the three whose weakness survives the correction.

Two caveats to keep this honest:

- October's per-day figure rests on 27 days rather than 31, so it is the noisiest of the twelve.
  The ranking flip is real; the size of the lead over November (\$2,371 against \$2,346) is not
  something to lean on.
- Whether those four Mondays were closures or a recording failure changes what to do next, and
  this dataset cannot say which. That question goes to the business, not to the data.

The practical effect is that a promotion aimed at October would have been aimed at the wrong
month. Section 13 recommends nothing on the strength of the monthly totals alone.

## 12. Insights

Total for 2015: **21,350 orders**, **49,574 pizzas**, **\$817,860** in revenue over **358 trading
days** - an average of **\$2,285 per day**. A typical order is **2.32 pizzas** worth **\$38.31**.

### Popular and Unpopular Pizzas

**Best sellers (units):** The Classic Deluxe (2,453), The Barbecue Chicken (2,432),
The Hawaiian (2,422), The Pepperoni (2,418), The Thai Chicken (2,371).

**Worst sellers (units):** The Brie Carre (490), The Mediterranean (934), The Calabrese (937),
The Spinach Supreme (950), The Soppressata (961).

The Brie Carre stands out: it sells roughly half as much as the next-worst pizza. A narrow
listing is not the explanation - it is one of three items sold in a single size, and the other two
(The Big Meat, S only; The Five Cheese, L only) move 1,914 and 1,409 units. Price is the more likely cause: the
cheapest size it can be bought in costs \$23.65, against a menu-wide average entry price of
\$12.79 and a next-highest of \$18.50.

### Sales by Hour

Trade is concentrated in two peaks - lunch (12:00-13:00) and dinner (17:00-19:00).

- **12:00-13:00** brings **\$217,944**, or **26.6%** of annual revenue, out of two hours a day.
- **17:00-19:00** adds **\$248,163** (30.3%).
- **09:00, 10:00 and 23:00 combined** produce **\$1,508 for the entire year** - **0.18%** of
  revenue, roughly 108x below the average hour.

09:00 is not a slow hour so much as a rounding error: across the whole year it contains **exactly
one order** - #19176, placed at 09:52 on 2015-11-24, four pizzas for \$83.

### Sales by Day of the Week

| Day | Revenue | Trading days | Revenue per trading day |
|---|---:|---:|---:|
| Friday | \$136,074 | 50 | \$2,721 |
| Thursday | \$123,529 | 52 | \$2,376 |
| Saturday | \$123,182 | 52 | \$2,369 |
| Monday | \$107,330 | 48 | \$2,236 |
| Wednesday | \$114,408 | 52 | \$2,200 |
| Tuesday | \$114,134 | 52 | \$2,195 |
| Sunday | \$99,204 | 52 | \$1,908 |

Friday is the clear best day, **16.5%** above the weekly average.

Monday ranks second-worst on total revenue, but it traded on four fewer days than the rest of
the week: the dataset is missing **all four Mondays in October**, so Monday is measured over 48
trading days instead of 52. Per trading day it sits mid-table, in fourth. **Sunday is the only
genuinely weak day** - **15.1%** below average on totals, and last on revenue per trading day as
well.

### Sales by Month

Revenue is flat across the year: every month lands between **\$64,028** and **\$72,558** against
an average of **\$68,155**, a spread of about +/-6%.

Those totals are misleading, and section 11 works through why. Three months traded on fewer days
than the calendar allows, and correcting for it reverses the bottom of the table. **October goes
from last on totals to first on revenue per trading day** (\$2,371), because all four of its
Mondays are absent from the data. September moves from 11th to 6th. Only **December is weak on
both measures** - 10th on totals, 12th per trading day.

**There is still no usable seasonal signal.** Per trading day the twelve months span \$2,157 to
\$2,371 - under 10%, with no shape to it, on a single year of data.

### Category and Size

| Category | Units | Revenue | Share |
|---|---:|---:|---:|
| Classic | 14,888 | \$220,053 | 26.9% |
| Supreme | 11,987 | \$208,197 | 25.5% |
| Chicken | 11,050 | \$195,920 | 24.0% |
| Veggie | 11,649 | \$193,690 | 23.7% |

The four categories are effectively tied. Classic leads on units by a wider margin than on
revenue because it is the cheapest category per pizza.

| Size | Units | Revenue | Share |
|---|---:|---:|---:|
| L | 18,956 | \$375,319 | 45.9% |
| M | 15,635 | \$249,382 | 30.5% |
| S | 14,403 | \$178,077 | 21.8% |
| XL | 552 | \$14,076 | 1.7% |
| XXL | 28 | \$1,007 | 0.1% |

Size is where the real concentration is: **L alone is 46% of revenue**, and L, M and S together
are **98.2%**. XL is offered on one pizza only (The Greek), XXL likewise, and XXL sold **28 units
in a year** - about one every two weeks.

## 13. Recommendations

### 1. Size operations around the lunch and dinner peaks
12:00-13:00 and 17:00-19:00 are 57% of revenue in five hours a day. Staffing, prep and delivery
capacity should be planned around those windows first; everything else is a secondary constraint.
Combo and pre-order offers work best aimed just *before* each peak (11:00 and 16:00), to flatten
the load rather than add to it.

### 2. Trim the opening hours at both ends
09:00, 10:00 and 23:00 generate \$1,508 a year between them - 0.18% of revenue, below the cost of
having anyone in the building. Opening at 11:00 and closing at 22:00 would give up that 0.18% and
remove roughly three staffed hours a day.

The caveat: this dataset has no cost, staffing or wage data, so the argument rests on revenue
alone. It is strong enough to justify a trial - close those hours for a quarter and check whether
the demand shifts to 11:00 or simply disappears.

### 3. Target Sunday, the weakest trading day
Sunday runs 15.1% below the weekly average on totals and is last on revenue per trading day as
well. It is the clearest candidate for a family or bundle offer.

Monday is a different case and should not be treated the same way. It looks weak on totals only
because four of its shifts are missing from the data; per trading day it is mid-table. Confirm
whether those four October Mondays were genuine closures or a recording gap before deciding
whether Monday needs anything at all.

### 4. Drop XXL, and review XL
XXL sold 28 units all year (\$1,007) and exists on a single pizza. It costs menu space, prep
training and inventory for 0.1% of revenue. XL is only marginally better at 1.7%.

The alternative reading is that both sizes are offered on one pizza only and so never had a fair
test. If the intent is to keep large formats, extend XL to the top five sellers and measure
again; if not, remove both and simplify the menu.

### 5. Re-price The Brie Carre, or drop it
At 490 units it is the slowest item on the menu, and the obvious excuse does not hold: it is not
handicapped by being sold in one size, because The Big Meat is also S-only and moves 1,914
units. What is unusual is the price. Its only size is a small at \$23.65, while the average entry
price on this menu is \$12.79 and the next dearest is \$18.50 - it is priced above every large on
the menu and sold as a small.

It also carries five ingredients that appear on no other pizza (brie carre cheese, prosciutto,
caramelised onions, pears, thyme), so it holds dedicated inventory for the slowest-moving dish.
Either test it at a price closer to its peers, or remove it and free that inventory.

### 6. Do not build a seasonal calendar yet, and do not act on the monthly totals
The monthly totals do not mean what they look like. October reads as the worst month of the year
and is in fact the best per trading day - the gap is four missing Mondays, not demand (section 11).
A promotion aimed at the weakest month would have been aimed at the strongest one.

Once coverage is accounted for, the twelve months span under 10% per trading day with no shape to
them. December is the only month whose weakness survives the correction, and one year is not
enough to say whether that repeats. A seasonal campaign calendar needs at least two more years of
data.

### What this dataset cannot answer
There are no customer identifiers (so no repeat rate, basket affinity or cohort analysis), no
costs or margins (so "most profitable" here means *highest revenue*, not highest profit), no
staffing levels or delivery times, and no promotion history. The staffing and menu
recommendations above are directional until cost data is available.

---

In [230]:
con.close()
print("done")

done
